# Part C: Prompt Engineering
**Student:** Dabone Abdoul Latif  
**Index:** 10012200015

This notebook focuses on designing prompt templates and testing them with the Groq LLM using the retrieval system from Part B.

### 1. Setup and Environment

In [8]:
import json
import faiss
import os
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from groq import Groq
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)

### 2. Load Processed Data Chunks

In [9]:
with open('chunks/csv_chunks.json', 'r') as f:
    csv_chunks = json.load(f)
with open('chunks/pdf_chunks.json', 'r') as f:
    pdf_chunks = json.load(f)

all_chunks = csv_chunks + pdf_chunks
print(f"Total chunks loaded: {len(all_chunks)}")

Total chunks loaded: 992


### 3. Initialize Retrieval Models

In [10]:
# Load embedding model and FAISS index
model = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index('indexes/rag_index.faiss')

# Setup BM25 keyword search
texts = [c['text'] for c in all_chunks]
tokenized_corpus = [text.lower().split() for text in texts]
bm25 = BM25Okapi(tokenized_corpus)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1789.57it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 4. Hybrid Search Function

In [11]:
def hybrid_search(query, k=3, alpha=0.5):
    # Semantic Search
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, k * 2)
    
    # Keyword Search
    bm25_scores = bm25.get_scores(query.lower().split())
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1
    
    combined = []
    for i, chunk in enumerate(all_chunks):
        v_score = 0
        for rank, idx in enumerate(indices[0]):
            if idx == i:
                v_score = 1 / (1 + distances[0][rank])
                break
        
        # Combine scores
        score = (alpha * v_score) + ((1 - alpha) * (bm25_scores[i] / max_bm25))
        if score > 0: 
            combined.append({'chunk': chunk, 'score': score})
            
    return sorted(combined, key=lambda x: x['score'], reverse=True)[:k]

### 5. Prompt Template Definitions

In [12]:
def prompt_v1(context, query):
    """Version 1: Basic structured prompt"""
    return f"""Use the following context to answer the question.
If the answer is not in the context, say you do not know.

Context:
{context}

Question: {query}
Answer:"""

def prompt_v2(context, query):
    """Version 2: Direct and concise instruction"""
    return f"""Answer the question based only on the provided documents.
Be direct and concise.

Documents:
{context}

User Query: {query}
Answer:"""

### 6. Context Management (Truncation)

In [13]:
def build_context(results, max_chars=4000):
    """Combine top chunks and truncate to fit token limits"""
    context = ""
    for res in results:
        c = res['chunk']
        context += f"\nSource: {c['source']}\n{c['text']}\n"
    return context[:max_chars]

### 7. Run Experiments

In [14]:
def run_experiment(query):
    results = hybrid_search(query)
    context = build_context(results)
    
    print(f"Testing Query: {query}\n")
    
    for p_func in [prompt_v1, prompt_v2]:
        prompt = p_func(context, query)
        res = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        print(f"--- Output from {p_func.__name__} ---")
        print(res.choices[0].message.content)
        print("\n")

# Example Test
run_experiment("What was the NDC vote percentage in the Volta region in 2020?")

Testing Query: What was the NDC vote percentage in the Volta region in 2020?

--- Output from prompt_v1 ---
The NDC vote percentage in the Volta region in 2020 was 84.83%.


--- Output from prompt_v2 ---
84.83%


